# 01 - Keşifsel Veri Analizi (EDA)

Bu notebook'ta Kronik Böbrek Hastalığı veri seti okunur ve modellemeden önce veri tanınır: sınıf dağılımı, eksik değerler ve sınıflar arasındaki farklar. 2. hafta raporundaki sayılar bu notebook'tan gelir.

In [ ]:
import sys
sys.path.append('..')   # src klasörünü görebilmek için

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_ckd

pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')

## 1. Veriyi okuma

Veri seti ARFF formatında. `load_ckd` fonksiyonu dosyadaki format hatalarını (fazladan virgül, tab karakterleri) düzeltip tabloya çevirir.

In [ ]:
df = load_ckd('../data/chronic_kidney_disease_full.arff')
print('Boyut:', df.shape)
df.head()

In [ ]:
df.info()

## 2. Sınıf dağılımı

- **ckd**: chronic kidney disease (hasta)
- **notckd**: not chronic kidney disease (sağlıklı)

In [ ]:
counts = df['class'].value_counts()
print(counts)
print((counts / len(df) * 100).round(1))

counts.plot(kind='bar', color=['#c0504d', '#4f81bd'])
plt.title('Sınıf dağılımı')
plt.ylabel('Kayıt sayısı')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../figures/sinif_dagilimi.png', dpi=150)
plt.show()

## 3. Eksik değerler

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing_table = pd.DataFrame({'eksik': missing, 'yuzde': (missing / len(df) * 100).round(1)})
missing_table[missing_table['eksik'] > 0]

In [ ]:
toplam_hucre = df.shape[0] * df.shape[1]
print('Toplam eksik hücre:', int(missing.sum()), f'(%{missing.sum() / toplam_hucre * 100:.1f})')
print('Hiç eksiği olmayan kayıt sayısı:', len(df.dropna()))

In [ ]:
missing_table.query('eksik > 0')['yuzde'].plot(kind='barh', color='#4f81bd', figsize=(7, 6))
plt.gca().invert_yaxis()
plt.title('Sütunlara göre eksik değer oranı (%)')
plt.xlabel('%')
plt.tight_layout()
plt.savefig('../figures/eksik_degerler.png', dpi=150)
plt.show()

## 4. Sınıflara göre laboratuvar değerleri

Ortalama yerine medyan kullanıyoruz, çünkü medyan aykırı değerlerden daha az etkilenir.

In [ ]:
lab = ['sc', 'bu', 'hemo', 'pcv', 'bgr', 'sod']
df.groupby('class')[lab].median().T

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, col, title in zip(axes, ['sc', 'bu', 'hemo', 'pcv'],
                          ['Serum kreatinin', 'Kan üresi', 'Hemoglobin', 'Hematokrit']):
    sns.boxplot(data=df, x='class', y=col, ax=ax, hue='class', palette=['#c0504d', '#4f81bd'], legend=False)
    ax.set_title(title)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig('../figures/lab_degerleri_boxplot.png', dpi=150)
plt.show()

## 5. Kategorik değişkenler

In [ ]:
for col in ['htn', 'dm', 'al', 'sg']:
    print(pd.crosstab(df[col], df['class']), '\n')

## 6. Bulgular

- 400 kayıt: 250 ckd (%62,5), 150 notckd (%37,5) → sınıflar dengesiz.
- Eksik hücre oranı %10,1; tam dolu kayıt sadece 158 → satır silmek yerine eksik değer doldurulacak.
- En çok eksik: rbc, rbcc, wbcc, pot, sod.
- Hastalarda kreatinin ve üre yüksek, hemoglobin ve hematokrit düşük.
- Sağlıklılarda hipertansiyon/diyabet hiç yok; albümin > 0 sadece hastalarda → sınıflar kolay ayrılıyor, aşırı öğrenmeye dikkat.